In [1]:
import torch
import torch.nn as nn
import time
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os

from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, IterableDataset

from architectures.gpt_wrapper import GPTWrapper
# from architectures.mamba_wrapper import MambaWrapper
from architectures.flb_transformer import FLB_Transformer
from architectures.fam_wrapper import FAMTransformer
# from architectures.xl_wrapper import XLWrapper

In [2]:
# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Settings
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB_SIZE = 50257 # GPT-2 Default
HIDDEN_DIM = 256
SEQ_LEN = 128
BATCH_SIZE = 16
MAX_TOKENS = 5_000_000 # Stop training after 5M tokens for the baseline test

# Data Structure for Benchmarking
results = {
    "model_name": [],
    "tokens_processed": [],
    "wall_clock_time": [],
    "loss": []
}

In [3]:
def log_metric(model_name, tokens, start_time, loss):
    results["model_name"].append(model_name)
    results["tokens_processed"].append(tokens)
    results["wall_clock_time"].append(time.time() - start_time)
    results["loss"].append(loss)

In [4]:
class WikiStreamDataset(IterableDataset):
    def __init__(self, dataset_name="wikitext-2-v1", split="train", seq_len=128):
        self.tokenizer = AutoTokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.dataset = load_dataset("wikitext", dataset_name, split=split, streaming=True)
        self.seq_len = seq_len

    def __iter__(self):
        buffer = []
        for example in self.dataset:
            text = example['text']
            if not text.strip(): continue
            tokens = self.tokenizer.encode(text)
            buffer.extend(tokens)
            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len:]
                yield (torch.tensor(chunk[:-1]), torch.tensor(chunk[1:]))

# Initialize the stream
train_data = WikiStreamDataset(dataset_name="wikitext-2-v1", seq_len=SEQ_LEN)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE)

In [ ]:
def run_benchmark(model, model_name, loader, checkpoint_interval=1_000_000):
    torch.cuda.empty_cache()
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
    criterion = nn.CrossEntropyLoss()
    
    # 1. CREATE STATIC BUFFERS (Required for CUDA Graphs)
    # These stay at fixed memory addresses so the Graph can find them
    static_input = torch.zeros((BATCH_SIZE, SEQ_LEN), dtype=torch.long, device=DEVICE)
    static_target = torch.zeros((BATCH_SIZE, SEQ_LEN), dtype=torch.long, device=DEVICE)
    
    # --- WARMUP & CAPTURE (Only for your FLB model) ---
    use_graphs = "FLB" in model_name
    graph = None
    static_loss = torch.tensor(0.0, device=DEVICE)

    if use_graphs:
        print(f"Warming up and auto-tuning Triton for {model_name}...")
        # Increase warmup to 20 to ensure all Triton tuning is finished
        for _ in range(20):
            optimizer.zero_grad(set_to_none=True)
            logits, aux = model(static_input)
            loss = criterion(logits.view(-1, logits.size(-1)), static_target.view(-1)) + aux
            loss.backward()
        
        # Crucial: Sync before capture to ensure GPU is totally idle
        torch.cuda.synchronize()
        
        print(f"Capturing CUDA Graph...")
        graph = torch.cuda.CUDAGraph()
        # Use a separate stream for capture to keep the recording "clean"
        capture_stream = torch.cuda.Stream()
        with torch.cuda.stream(capture_stream):
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.graph(graph):
                static_logits, static_aux = model(static_input)
                loss_main = criterion(static_logits.view(-1, static_logits.size(-1)), static_target.view(-1))
                static_loss = loss_main + static_aux
                static_loss.backward()
        torch.cuda.synchronize()

    # --- TRAINING LOOP ---
    start_time = time.time()
    tokens_count = 0
    pbar = tqdm(total=MAX_TOKENS, desc=f"Training {model_name}")
    
    for x, y in loader:
        # Copy data into static buffers
        static_input.copy_(x.to(DEVICE))
        static_target.copy_(y.to(DEVICE))
        
        optimizer.zero_grad(set_to_none=True)
        
        if graph:
            graph.replay() # Execute all 260 wavefront steps instantly
            optimizer.step()
            current_loss = static_loss.item()
        else:
            # Normal execution for other baseline models
            logits = model(static_input)
            loss = criterion(logits.view(-1, logits.size(-1)), static_target.view(-1))
            loss.backward()
            optimizer.step()
            current_loss = loss.item()

        # 2. UPDATE PROGRESS BAR WITH LIVE LOSS
        tokens_count += x.numel()
        log_metric(model_name, tokens_count, start_time, current_loss)
        
        # Only update the bar UI occasionally to save CPU cycles
        if tokens_count % (BATCH_SIZE * 10) == 0:
            pbar.set_postfix(loss=f"{current_loss:.4f}", refresh=True)
        
        pbar.update(x.numel())
        if tokens_count >= MAX_TOKENS: break

    pbar.close()
    return results

In [6]:
# 1. FLB-Transformer (Your Model)
flb = FLB_Transformer(vocab_size=VOCAB_SIZE, hidden_dim=HIDDEN_DIM, num_layers=6, seq_len=SEQ_LEN, batch_size=BATCH_SIZE)
run_benchmark(flb, "FLB-Transformer", train_loader)

Capturing CUDA Graph for FLB-Transformer...


RuntimeError: CUDA error: operation failed due to a previous error during capture
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# 2. GPT Baseline
gpt = GPTWrapper(VOCAB_SIZE, HIDDEN_DIM, num_layers=6, nhead=8, seq_len=SEQ_LEN)
run_benchmark(gpt, "GPT-Baseline", train_loader)

In [ ]:
# 3. Mamba Baseline
mamba = MambaWrapper(VOCAB_SIZE, HIDDEN_DIM, num_layers=6)
run_benchmark(mamba, "Mamba-SSM", train_loader)

In [ ]:
# 4. FAM Baseline
fam = FAMTransformer(VOCAB_SIZE, HIDDEN_DIM, num_layers=6)
run_benchmark(fam, "FAM-Baseline", train_loader)

In [ ]:
# 5. XL Baseline
xl = XLWrapper(VOCAB_SIZE, HIDDEN_DIM, num_layers=6)
run_benchmark(xl, "XL-Baseline", train_loader)

In [ ]:
all_run_data = []

# Example run
flb_results = run_benchmark(flb, "FLB-Transformer", train_loader)
all_run_data.extend(flb_results)

# Then at the end of your notebook
df = pd.DataFrame(all_run_data)